# TUGAS PRAKTIKUM 5 KECERDASAN KOMPUTASIONAL

| Informasi Mahasiswa | |
| :--- | :--- |
| **Nama** | Catherine Aprilia Harlina |
| **NRP** | 5054251012 |
| **Program Studi** | Rekayasa Kecerdasan Artifisial |
| **Mata Kuliah** | Kecerdasan Komputasional |

---

In [71]:
# Standard setup cell - run before other cells.
import importlib.util
import subprocess
import sys
from pathlib import Path

def ensure_dependencies():
    """Install only packages used by this module."""
    required = {"numpy": "numpy", "matplotlib": "matplotlib", "pandas": "pandas"}
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_module_dir(start):
    """Find the shared membership module."""
    for root in [start, *start.parents]:
        for candidate in (root, root / "Fuzzy Logic"):
            if (candidate / "membership.py").exists():
                return candidate
        if (root / ".git").exists():
            break
    return None

ensure_dependencies()
module_dir = find_module_dir(Path.cwd())
if module_dir is None and "google.colab" in sys.modules:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git", str(clone_dir)], check=True)
    module_dir = find_module_dir(clone_dir)
if module_dir is None:
    raise RuntimeError("membership.py not found; open this notebook inside the repository")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from membership import linear_up, linear_down, triangular, trapezoidal

print("Python:", sys.version.split()[0])
print("Membership module:", module_dir / "membership.py")

Python: 3.12.4
Membership module: c:\Users\LEGION\Praktikum-KK\fuzzy-logic\membership.py


# soal 1 - pengaturan kecepatan kipas tsukamoto

nomor 1.1 - cari semua nilai z yg mberi derajat 0,8 pd triangular(z, 25, 50, 75) trs jlsn knapa sgitiga gak bsa dipake jd konsekuen tsukamoto

nomor satu ni kita nyari nilai z dlu pas mu nya 0,8 di sgitiga (25, 50, 75) kan kaki ada dua kiri sma kanan:
- kaki kiri (naik): mu = (z - 25) / (50 - 25) = 0,8 -> z - 25 = 20 -> z = 45
- kaki kanan (turun): mu = (75 - z) / (75 - 50) = 0,8 -> 75 - z = 20 -> z = 55
nah dapet nilai z ny di 45 sma 55

trus alesan sgitiga gak bsa dipake di tsukamoto tu krn metode ny butuh fungsi konsekuen yg monoton (hrus satu arah aj naik atau turun). klo pake sgitiga ntar satu nilai mu bsa dapet dua nilai z yg beda (kyk 0,8 dapet 45 sma 55), jdinya ambigu gak jelas arah fungsi inversny. makanya diganti pke salah satu kaki aj kyk linear up atau linear down

In [72]:
from membership import triangular

mu_target = 0.8
a, b, c = 25, 50, 75
z_kiri = a + mu_target * (b - a)
z_kanan = c - mu_target * (c - b)

print(f"nilai z nya dapet {z_kiri} sm {z_kanan}")

nilai z nya dapet 45.0 sm 55.0


nomor 1.2 - hitung kecepatan kipas dket suhu 28c sma 36c pake smua tahap tsukamoto trus bikin alfa sm zi dlm tabel

a. suhu 28c:
1. fuzzifikasi & alfa (poin 1):
- sejuk = linear_down(28, 20, 30) = 0,2 -> alfa1 = 0,2
- nyaman = triangular(28, 20, 30, 40) = 0,8 -> alfa2 = 0,8
- panas = linear_up(28, 30, 40) = 0 -> alfa3 = 0
2. arah konsekuen & hitung zi pake rumus invers (poin 2):
- r1 (lambat, turun 0-50): z1 = 50 - 0,2(50) = 40
- r2 (sedang, naik 25-50): z2 = 25 + 0,8(25) = 45
- r3 (cepat, naik 50-100): z3 = 50 (krn alfa3 = 0)
3. sum alfa, sum alfa*zi, sma z* (poin 3):
- sum alfa = 0,2 + 0,8 + 0 = 1,0
- sum alfa*zi = (0,2 * 40) + (0,8 * 45) = 44
- z* = 44 / 1,0 = 44
4. cek rentang zi sma posisi z* (poin 4): smua zi di rentang miringny sma z* ad di tngh zi aktif (valid).
5. input pembanding (poin 5): ntar dbandingin sma suhu 36c dbwah.

tabel 28c:
| aturan | alfa | konsekuen | zi |
|---|---|---|---|
| r1 | 0,2 | lambat | 40 |
| r2 | 0,8 | sedang | 45 |
| r3 | 0,0 | cepat | 50 |

b. suhu 36c:
1. fuzzifikasi & alfa (poin 1):
- sejuk = 0 -> alfa1 = 0
- nyaman = 0,4 -> alfa2 = 0,4
- panas = 0,6 -> alfa3 = 0,6
2. arah konsekuen & zi (poin 2):
- r1: z1 = 50
- r2: z2 = 25 + 0,4(25) = 35
- r3: z3 = 50 + 0,6(50) = 80
3. sum alfa, sum alfa*zi, z* (poin 3):
- sum alfa = 1,0
- sum alfa*zi = 0 + 14 + 48 = 62
- z* = 62 / 1,0 = 62
4. cek rentang zi sma z* (poin 4): valid ad di rentang.
5. input pembanding (poin 5): pas suhu naik dr 28c k 36c, aturan brgeser jdi bnyk k cepat (z3 = 80) jdinya z* naek signifikan dr 44% k 62%.

tabel 36c:
| aturan | alfa | konsekuen | zi |
|---|---|---|---|
| r1 | 0,0 | lambat | 50 |
| r2 | 0,4 | sedang | 35 |
| r3 | 0,6 | cepat | 80 |

In [73]:
from membership import linear_up, linear_down, triangular
import pandas as pd
from tabulate import tabulate

def inverse_linear_down(alpha, a, b):
    return b - alpha * (b - a)

def inverse_linear_up(alpha, a, b):
    return a + alpha * (b - a)

def weighted_average(alphas, zs):
    total_alpha = sum(alphas)
    if total_alpha == 0:
        return 0
    return sum(a * z for a, z in zip(alphas, zs)) / total_alpha

def get_tsukamoto_table(t):
    mu_s = linear_down(t, 20, 30)
    mu_n = triangular(t, 20, 30, 40)
    mu_p = linear_up(t, 30, 40)
    
    alphas = [mu_s, mu_n, mu_p]
    
    z1 = inverse_linear_down(alphas[0], 0, 50)
    z2 = inverse_linear_up(alphas[1], 25, 50)
    z3 = inverse_linear_up(alphas[2], 50, 100)
    zs = [z1, z2, z3]
    
    z_star = weighted_average(alphas, zs)
    
    df = pd.DataFrame({
        "aturan": ["r1 (sejuk -> lambat)", "r2 (nyaman -> sedang)", "r3 (panas -> cepat)"],
        "alfa_i": [f"{a:.1f}" for a in alphas],
        "konsekuen": ["lambat", "sedang", "cepat"],
        "z_i": [f"{z:.1f}" for z in zs],
        "alfa_z": [f"{a * z:.1f}" for a, z in zip(alphas, zs)]
    })
    
    print(f"tabel kluaran bwt suhu {t}c:")
    print(tabulate(df, headers='keys', tablefmt='github', showindex=False))
    print(f"z* (rata-rata berbobot): {z_star:.2f}\n")

get_tsukamoto_table(28)
get_tsukamoto_table(36)

tabel kluaran bwt suhu 28c:
| aturan                |   alfa_i | konsekuen   |   z_i |   alfa_z |
|-----------------------|----------|-------------|-------|----------|
| r1 (sejuk -> lambat)  |      0.2 | lambat      |    40 |        8 |
| r2 (nyaman -> sedang) |      0.8 | sedang      |    45 |       36 |
| r3 (panas -> cepat)   |      0   | cepat       |    50 |        0 |
z* (rata-rata berbobot): 44.00

tabel kluaran bwt suhu 36c:
| aturan                |   alfa_i | konsekuen   |   z_i |   alfa_z |
|-----------------------|----------|-------------|-------|----------|
| r1 (sejuk -> lambat)  |      0   | lambat      |    50 |        0 |
| r2 (nyaman -> sedang) |      0.4 | sedang      |    35 |       14 |
| r3 (panas -> cepat)   |      0.6 | cepat       |    80 |       48 |
z* (rata-rata berbobot): 62.00



nomor 3 - bandingin hasilnya sma mamdani sma sugeno orde nol trs jlsn sumber perbedaanny

markdown manual

intinya beda metode beda cara nentuin output sma defuzzifikasiny:
- tsukamoto pke fungsi monoton trs zi dicari pke rumus invers dr alfa
- mamdani pke himpunan fuzzy asli trs defuzzifikasi ny pke luasan area centroid.
- sugeno orde nol pke konstanta tegas bwt konsekuenny.
makanya hasil akhir bs beda krn algoritmanya beda pas nerjemahin nilai alfa k output

In [74]:
# Cell Code - Analisis / Print Ringkasan Perbandingan Metode
metode_info = {
    "Tsukamoto": "Output berupa fungsi monoton, defuzzifikasi rata-rata terbobot dari z_i invers.",
    "Mamdani": "Output berupa himpunan fuzzy, defuzzifikasi menggunakan centroid/area.",
    "Sugeno Orde Nol": "Output berupa konstanta tegas (crisp value)."
}

for m, desc in metode_info.items():
    print(f"- {m}: {desc}")

- Tsukamoto: Output berupa fungsi monoton, defuzzifikasi rata-rata terbobot dari z_i invers.
- Mamdani: Output berupa himpunan fuzzy, defuzzifikasi menggunakan centroid/area.
- Sugeno Orde Nol: Output berupa konstanta tegas (crisp value).


nomor 4 - hitung kluaran tsukamoto suhu 10-45c step 0,1c trs cek apkh sllu naek klo suhu naek, kasi tau alesan dr alfa sma zi

markdown manual

nah bwt yg nomor 4 ni kn kita disuruh ngitung kluaran z* dlm rentang suhu bnyak bgt dr 10c smpe 45c dngan step 0,1c, jdi totalny ad skitar 351 titik suhu bgtukan. jdi cara manualny bkn brati kita ngitungin satu-satu manual pake angka manual sebanyak itu, tpi kita jabarinn konsep step by step alur pengerjaan yg dilakuin komputernya di stiap titikny.

jadi d stiap titik suhu t dlm rentang itu, prosesny jalan gni:
1. hitung dlu derajat keanggotaan mu bwt stiap label pake fungsi input sperti biasa (sejuk, nyaman, panas)
2. dari mu itu, kita dapet nilai alfa i bwt stiap aturan
3. trus cari nilai zi bwt stiap aturan pake fungsi invers (inverse_linear_down bwt lambat, inverse_linear_up bwt sedang sma cepat)
4. hitung kluaran stengah jadianny pake rumus rata-rata berbobot, yitu z* = sum(alfa * zi) / sum(alfa)
5. nah dr hasil simulasi perulangan itu, pas di cek trnyata kluaran z* ny **gak sllu naik** mulus trs lurus keatas, kdg d bbrp titik transisi ad kelokan kecil atw penurunan bgtu
6. alibi atw penyebab dr alfa sma zi knapa bsa gt krn ad efek overlapping antarkurva keanggotaan. pas suhu geser d daerah transisi, kombinasi perubahan alfa yg dkali sm zi trs dibagi sum alfa bsa nimbulin fluktuasi tipis sblm akhirny ntar naek lg ngikutin suhu

In [75]:
import numpy as np

suhu_arr = np.arange(10, 45.1, 0.1)
hasil_simulasi = []

for t in suhu_arr:
    t_round = round(t, 1)
    alphas, zs, z_star = run_tsukamoto(t_round)
    hasil_simulasi.append((t_round, z_star))

# cek apkh sllu naek
z_values = [item[1] for item in hasil_simulasi]
penurunan = np.any(np.diff(z_values) < 0)

print(f"total data suhu: {len(suhu_arr)}")
print(f"apakah kluaran selalu naik? {'gak, ada bbrp yg turun/fluktuasi' if penurunan else 'ya, selalu naik'}")

# nampilin bbrp smpel kluaran d rentang tertentu
print("\nsampel bbrp kluaran z*:")
for t, z in hasil_simulasi[::50]: # ambil tiap 5 derajat sbg smpel
    print(f"suhu {t}c -> kluaran z* = {z:.2f}")

total data suhu: 351
apakah kluaran selalu naik? gak, ada bbrp yg turun/fluktuasi

sampel bbrp kluaran z*:
suhu 10.0c -> kluaran z* = 0.00
suhu 15.0c -> kluaran z* = 0.00
suhu 20.0c -> kluaran z* = 0.00
suhu 25.0c -> kluaran z* = 31.25
suhu 30.0c -> kluaran z* = 50.00
suhu 35.0c -> kluaran z* = 56.25
suhu 40.0c -> kluaran z* = 100.00
suhu 45.0c -> kluaran z* = 100.00


In [76]:
# SOAL 2